# Regeneraci?n reproducible de figuras del manuscrito

Este notebook usa las tablas consolidadas incluidas en el paquete. No entrena modelos ni modifica resultados experimentales.


In [ ]:
# Dependencias
import importlib.util, subprocess, sys
required = {
    'numpy': 'numpy>=1.24',
    'pandas': 'pandas>=2.0',
    'matplotlib': 'matplotlib>=3.7',
    'PIL': 'Pillow>=10.0',
    'reportlab': 'reportlab>=4.0',
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencias listas.')


In [ ]:
# Localizar la ra?z del paquete desde cualquier subcarpeta
from pathlib import Path

def locate_package_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        script = candidate / '02_CODIGO' / 'paper_figures' / 'generate_paper_figures.py'
        if script.exists():
            return candidate
    for script in start.resolve().rglob('generate_paper_figures.py'):
        if script.parent.name == 'paper_figures':
            return script.parents[2]
    raise FileNotFoundError('No se encontr? la ra?z de ENTREGA_FINAL_CLASIFICACION_SELECTIVA_ECG_20260728.')

PACKAGE_ROOT = locate_package_root(Path.cwd())
FIG_SCRIPT = PACKAGE_ROOT / '02_CODIGO' / 'paper_figures' / 'generate_paper_figures.py'
DATA_DIR = PACKAGE_ROOT / '03_RESULTADOS' / 'paper_source_tables'
OUT_DIR = PACKAGE_ROOT / '05_FIGURAS' / 'manuscrito'
print('PACKAGE_ROOT:', PACKAGE_ROOT)


In [ ]:
# Auditar insumos
import pandas as pd
required = [
    DATA_DIR / 'calibration_class_diffs.csv',
    DATA_DIR / 'review_metrics.csv',
    DATA_DIR / 'enrichment_summary.csv',
    DATA_DIR / 'class_capture_summary.csv',
    DATA_DIR / 'class_bootstrap_diffs.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Faltan tablas: ' + '\n'.join(missing)
print('Tablas verificadas:', len(required))
display(pd.read_csv(DATA_DIR / 'review_metrics.csv').head())


In [ ]:
# Regenerar las seis figuras del manuscrito
import runpy
runpy.run_path(str(FIG_SCRIPT), run_name='__main__')


In [ ]:
# Verificar y mostrar resultados
from PIL import Image
import matplotlib.pyplot as plt

expected = [
    'fig01_diseno_estudio',
    'fig02_calibracion_por_clase',
    'fig03_auprc_revision',
    'fig04_enriquecimiento_baja_certeza',
    'fig05_captura_baja_certeza_vs_error',
    'fig06_captura_error_por_clase',
]
missing = [str(OUT_DIR / f'{stem}{ext}') for stem in expected for ext in ('.png', '.pdf')
           if not (OUT_DIR / f'{stem}{ext}').exists()]
assert not missing, 'Faltan salidas: ' + '\n'.join(missing)
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
for ax, stem in zip(axes.flat, expected):
    ax.imshow(Image.open(OUT_DIR / f'{stem}.png'))
    ax.set_title(stem)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Figuras verificadas en:', OUT_DIR)
